# Natural Language Processing with Disaster Tweets
**Predict which Tweets are about real disasters and which ones are not**

***Acknowledgments***
This dataset was created by the company figure-eight and originally shared on their ‘Data For Everyone’ website here.

Tweet source: https://twitter.com/AnyOtherAnnaK/status/629195955506708480

Kaggle source: https://www.kaggle.com/competitions/nlp-getting-started/overview

In [53]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import re
import nltk
import string
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer, PorterStemmer
from sklearn.model_selection import train_test_split

nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/festusattornelson/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/festusattornelson/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

## Step 1. Data Exploration

In [54]:
df = pd.read_csv('/Users/festusattornelson/Documents/Projects/Python_Udemy/Projects/NLP-disastertweets/data/disaster_treets_train.csv')

In [55]:
df.head()

,id,keyword,location,text,target
0,1,NaN,NaN,Our Deeds are the Reason of this #earthquake M...,1
1,4,NaN,NaN,Forest fire near La Ronge Sask. Canada,1
2,5,NaN,NaN,All residents asked to 'shelter in place' are ...,1
3,6,NaN,NaN,"13,000 people receive #wildfires evacuation or...",1
4,7,NaN,NaN,Just got sent this photo from Ruby #Alaska as ...,1


In [56]:
df.shape

(7613, 5)

In [57]:
df_train =pd.DataFrame({'text':df['text'], 'target':df['target']})

In [58]:
df_train.head()

,text,target
0,Our Deeds are the Reason of this #earthquake M...,1
1,Forest fire near La Ronge Sask. Canada,1
2,All residents asked to 'shelter in place' are ...,1
3,"13,000 people receive #wildfires evacuation or...",1
4,Just got sent this photo from Ruby #Alaska as ...,1


In [59]:
df_train.target.value_counts()

target
0    4342
1    3271
Name: count, dtype: int64

In [60]:
df_train.info()

<class 'pandas.DataFrame'>
RangeIndex: 7613 entries, 0 to 7612
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   text    7613 non-null   str  
 1   target  7613 non-null   int64
dtypes: int64(1), str(1)
memory usage: 119.1 KB


In [61]:
df['char_count'] = df['text'].str.len()
df['word_count'] = df['text'].str.split().apply(len)
df['hashtag_count'] = df['text'].str.count(r'#\w+')
df['mention_count'] = df['text'].str.count(r'@\w+')
df['url_count'] = df['text'].str.count(r'http\S+')

df[['text','char_count','word_count','hashtag_count','mention_count','url_count']].head()

,text,char_count,word_count,hashtag_count,mention_count,url_count
0,Our Deeds are the Reason of this #earthquake M...,69,13,1,0,0
1,Forest fire near La Ronge Sask. Canada,38,7,0,0,0
2,All residents asked to 'shelter in place' are ...,133,22,0,0,0
3,"13,000 people receive #wildfires evacuation or...",65,8,1,0,0
4,Just got sent this photo from Ruby #Alaska as ...,88,16,2,0,0


## Step 2. Preprocessing

In [62]:
# Lowercasing
df_train['text'] = df_train['text'].apply(lambda text: text.lower())
df_train.head()

,text,target
0,our deeds are the reason of this #earthquake m...,1
1,forest fire near la ronge sask. canada,1
2,all residents asked to 'shelter in place' are ...,1
3,"13,000 people receive #wildfires evacuation or...",1
4,just got sent this photo from ruby #alaska as ...,1


In [63]:
# Cleaning - remove numbers
df_train['text'] = df_train['text'].apply(lambda text: re.sub(r'\d+', '', text))
df_train.head()

,text,target
0,our deeds are the reason of this #earthquake m...,1
1,forest fire near la ronge sask. canada,1
2,all residents asked to 'shelter in place' are ...,1
3,", people receive #wildfires evacuation orders ...",1
4,just got sent this photo from ruby #alaska as ...,1


In [64]:
# Removing hashtags and special characters
df_train['text'] = df_train['text'].apply(lambda text: re.sub(r'[^a-zA-Z\s]', '', text))
df_train.head()

,text,target
0,our deeds are the reason of this earthquake ma...,1
1,forest fire near la ronge sask canada,1
2,all residents asked to shelter in place are be...,1
3,people receive wildfires evacuation orders in...,1
4,just got sent this photo from ruby alaska as s...,1


In [65]:
# Removing punctuation
df_train['text'] = df_train['text'].apply(lambda text: text.translate(str.maketrans('', '', string.punctuation)))
df_train.head()

,text,target
0,our deeds are the reason of this earthquake ma...,1
1,forest fire near la ronge sask canada,1
2,all residents asked to shelter in place are be...,1
3,people receive wildfires evacuation orders in...,1
4,just got sent this photo from ruby alaska as s...,1


In [66]:
# Tokenization
df_train['text'] = df_train['text'].apply(lambda text: text.split())
df_train.head()

,text,target
0,"[our, deeds, are, the, reason, of, this, earth...",1
1,"[forest, fire, near, la, ronge, sask, canada]",1
2,"[all, residents, asked, to, shelter, in, place...",1
3,"[people, receive, wildfires, evacuation, order...",1
4,"[just, got, sent, this, photo, from, ruby, ala...",1


In [67]:
# Filtering out non-alphabetic characters and short tokens
df_train['text'] = df_train['text'].apply(lambda text: [word for word in text if word.isalpha() and len(word) > 1   ])
df_train.head()

,text,target
0,"[our, deeds, are, the, reason, of, this, earth...",1
1,"[forest, fire, near, la, ronge, sask, canada]",1
2,"[all, residents, asked, to, shelter, in, place...",1
3,"[people, receive, wildfires, evacuation, order...",1
4,"[just, got, sent, this, photo, from, ruby, ala...",1


In [68]:
# Stop words removal
stop_words = set(stopwords.words('english'))
df_train['text'] = df_train['text'].apply(lambda text: [word for word in text if word not in stop_words])
df_train.head()

,text,target
0,"[deeds, reason, earthquake, may, allah, forgiv...",1
1,"[forest, fire, near, la, ronge, sask, canada]",1
2,"[residents, asked, shelter, place, notified, o...",1
3,"[people, receive, wildfires, evacuation, order...",1
4,"[got, sent, photo, ruby, alaska, smoke, wildfi...",1


In [69]:
# Lemmatization
lemmatizer = WordNetLemmatizer()
df_train['text'] = df_train['text'].apply(lambda text: [lemmatizer.lemmatize(word) for word in text])
df_train.head()

,text,target
0,"[deed, reason, earthquake, may, allah, forgive...",1
1,"[forest, fire, near, la, ronge, sask, canada]",1
2,"[resident, asked, shelter, place, notified, of...",1
3,"[people, receive, wildfire, evacuation, order,...",1
4,"[got, sent, photo, ruby, alaska, smoke, wildfi...",1


In [70]:
df_train['text'] = df_train['text'].apply(lambda x: ' '.join(x))

## Step 3. Feature Extraction

### 3.1 Data Split

In [71]:
# import trian-test library
from sklearn.model_selection import train_test_split

In [72]:
X = df['text']
y = df['target']

In [73]:
# Split into training and test set
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2, random_state=42, stratify=y)

In [74]:
print(f"Training set size: {len(X_train)}")
print(f"Test set size: {len(X_test)}")

Training set size: 6090
Test set size: 1523


### 3.2 CountVectorizer Feature Extraction

In [75]:
# import count vectorizer
from sklearn.feature_extraction.text import CountVectorizer
countVect = CountVectorizer()

In [76]:
# Fit and transform only on train set
X_train_cv = countVect.fit_transform(X_train)

In [77]:
# Transform only on test set
X_test_cv = countVect.transform(X_test)

In [87]:
# Check the number of samples and features
n_samples, n_features = X_train_cv.shape
print("n_samples: {}, n_features: {}".format(n_samples, n_features))

n_samples: 6090, n_features: 18566


### 3.3 TfidfVectorizer Feature Extraction

In [79]:
# import TF-IDF Vectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf_vectorizer = TfidfVectorizer(max_features=5000)

In [83]:
# Fit and transform only on train set
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)

In [84]:
# Transform only on test set
X_test_tfidf = tfidf_vectorizer.transform(X_test)

In [85]:
# Check the number of samples and features
num_samples, num_features = X_train_tfidf.shape
print("num_samples: {}, num_features: {}".format(num_samples, num_features))

num_samples: 6090, num_features: 5000


In [ ]:
## Compare the results from both vectorizers

## Step 4. Model Training and Evaluation

### 4.1 CountVectorizer Model Evaluation

### 4.2 Tfidfectorizer Model Evaluation

## Step 5. Hyperparameter Tuning

## Step 6. Check incorrect predictions